# ADMET-AI CPU example
Supports ADMET properties and toxicity prediction of compounds.
## Overview
- **Field** — ADMET prediction
- **Category** — Screening · Property
- **Model type** — Chemprop (GNN)
- **Year** — 2024
- **License** — MIT
**Related papers**

- [ADMET-AI: a machine learning ADMET platform for evaluation of large-scale chemical libraries](https://academic.oup.com/bioinformatics/article/40/7/btae416/7698030) — Kyle Swanson et al., Bioinformatics, 2024

> **CPU example.** This notebook uses a single SMILES or a short peptide input. Its input workflow, package version, and output types may differ from the platform.

**Notebook package version:** 2.0.1. **Platform catalog version:** 1.3.1. Do not assume predictions from different versions are equivalent. Installed versions are recorded with the outputs.

## Requirements

- **Repository** — https://github.com/swansonk14/admet_ai
- **Libraries** — see the environment section below
- **Run it on the platform** — https://lab.curieus.net/Analysis/admet-ai

## Limits and cautions
- Runs on CPU alone — no GPU runtime needed.
- Time grows with the number of molecules. Split batches of a few thousand.
- When the session ends, installed packages and downloaded weights are gone. Reconnecting means installing again.
- The free runtime disconnects after a period of inactivity. For long runs, saving intermediate results to Google Drive is safer.

## Procedure

Use a CPU runtime and run the setup, input, inference, and results cells in order. A GPU is not required.

This example predicts properties for aspirin using ADMET-AI v2. It does not reproduce results from the paper or the platform's v1 model.

## Set up the environment

In [ ]:
import os, sys, platform, subprocess
from pathlib import Path
from sysconfig import get_path
SCRIPTS = Path(get_path("scripts"))
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # CPU example
os.environ['MPLBACKEND'] = 'Agg'  # CLI runs outside the notebook kernel
print('Python:', platform.python_version())
subprocess.run([sys.executable, "-m", "pip", "install", "admet-ai==2.0.1"], check=True)

## Set the inputs

In [ ]:
SMILES = "CC(=O)Oc1ccccc1C(=O)O"  # @param {type:"string"}
from pathlib import Path
import pandas as pd
from rdkit import Chem
if Chem.MolFromSmiles(SMILES) is None:
    raise ValueError("Invalid SMILES")
pd.DataFrame({"smiles": [SMILES]}).to_csv("input.csv", index=False)

## Run

In [ ]:
from pathlib import Path
Path("preds.csv").unlink(missing_ok=True)  # never display an old result after a failed run
subprocess.run([str(SCRIPTS / "admet_predict"), "--data_path", "input.csv", "--save_path", "preds.csv", "--smiles_column", "smiles"], check=True, timeout=600)

## Check the results

In [ ]:
import json
import pandas as pd
from importlib.metadata import version
from IPython.display import display
result_path = Path("preds.csv")
if not result_path.is_file() or result_path.stat().st_size == 0:
    raise RuntimeError("Prediction output was not created")
predictions = pd.read_csv(result_path)
expected_rows = 1
if len(predictions) != expected_rows or predictions.select_dtypes(include="number").dropna(how="all").empty:
    raise RuntimeError("Prediction output is empty or incomplete")
required_scores = ["AMES","hERG"]
import numpy as np
if not set(required_scores).issubset(predictions.columns):
    raise RuntimeError("Expected model prediction columns are missing")
scores = predictions[required_scores].apply(pd.to_numeric, errors="raise")
if not np.isfinite(scores.to_numpy()).all():
    raise RuntimeError("Model prediction scores contain missing or non-finite values")
display(predictions)
environment = {"python": platform.python_version(), "package": "admet-ai", "version": version("admet-ai"), "platform_catalog_version": "1.3.1"}
Path("environment.json").write_text(json.dumps(environment, indent=2))
print(environment)
print("Saved: preds.csv, environment.json")

## Interpreting the results

The output is a CSV of predicted properties. Probability and regression columns have different meanings and units; consult the repository documentation. These are not experimental measurements or a medicine safety assessment.

Consult the paper and repository for metric definitions.